# পাঠ ০৫ - এজেন্টিক RAG


## সেটআপ

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে Agentic RAG (Retrieval-Augmented Generation) প্যাটার্ন প্রদর্শন করে।

**প্রয়োজনীয়তা:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — আপনার Azure AI Search সেবার এন্ডপয়েন্ট
- `AZURE_SEARCH_API_KEY` — আপনার Azure AI Search API কী
- পরিবেশ ভেরিয়েবলগুলির মাধ্যমে কনফিগার করা Azure OpenAI ডিপ্লয়মেন্ট
- Azure CLI প্রমাণীকৃত (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## এজেন্টিক RAG কী?

ঐতিহ্যবাহী RAG একটি নির্দিষ্ট ধারা অনুসরণ করে: ডকুমেন্টগুলি অনুসন্ধান করা, তারপরে একটি উত্তর তৈরি করা। **এজেন্টিক RAG** আরও এগিয়ে যায় এজেন্টকে স্বাধীনতা দিয়ে **কখন** এবং **কিভাবে** তথ্য অনুসন্ধান করতে হবে তা সিদ্ধান্ত নেয়ার জন্য।

এজেন্টিক RAG এর মাধ্যমে, এজেন্ট করতে পারে:
- প্রশ্নের উত্তরের আগে অনুসন্ধান প্রয়োজন কিনা তা **সিদ্ধান্ত নেওয়া**
- কোন তথ্য উৎস বা টুল কুয়েরি করা হবে তা **বাছাই করা**
- প্রাপ্ত ফলাফল মূল্যায়ন করে যদি প্রথম চেষ্টা অপর্যাপ্ত হয়, তবে পুনরায় অনুসন্ধান করা
- একাধিক অনুসন্ধান ধাপ থেকে তথ্য **মিলিয়ে** একটি সুসংগত উত্তর তৈরি করা

এটা এজেন্টকে একটি স্থির retrieve-then-generate ধারা থেকে অনেক বেশি নমনীয় এবং সঠিক করে তোলে।


## একটি অনুসন্ধান সরঞ্জাম তৈরি করা

Agentic RAG-এ, বাহ্যিক তথ্যসূত্রগুলোকে **সরঞ্জাম** হিসেবে মোড়ানো হয় যা এজেন্ট প্রয়োজনে ডাকার জন্য ব্যবহার করতে পারে। এটি এজেন্টকে অন্বেষণকে আরেকটি ক্রিয়াকলাপ হিসেবে বিবেচনা করতে দেয়, বাধ্যতামূলক ধাপ হওয়ার পরিবর্তে।

নিচে আমরা একটি ভ্রমণ জ্ঞানভিত্তি সংজ্ঞায়িত করেছি এবং এটি একটি সরঞ্জাম হিসেবে উপস্থাপন করেছি যা এজেন্ট গন্তব্য তথ্যlook করার জন্য ডেকে নিতে পারে।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG এজেন্ট তৈরি করা

এখন আমরা একটি এজেন্ট তৈরি করব যাকে নির্দেশ দেওয়া হয়েছে **উত্তর দেওয়ার আগে সবসময় তথ্য অনুসন্ধান করতে**। এজেন্ট তার উত্তর ভিত্তিভূত করার জন্য নিজের প্রশিক্ষণ ডেটার ওপর নির্ভর না করে `search_travel_knowledge` টুল ব্যবহার করে জ্ঞানের ভিত্তিতে কাজ করে।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ধাপে ধাপে তথ্য অনুসন্ধান — মেকার-চেকার প্যাটার্ন

Agentic RAG-এর একটি মূল সুবিধা হলো **ধাপে ধাপে তথ্য অনুসন্ধান**। এজেন্ট তার প্রাথমিক অনুসন্ধানের ফলাফল যাচাই, পরিমার্জন বা সম্প্রসারণ করতে একাধিক রাউন্ড অনুসন্ধান করতে পারে — যা একটি "মেকার-চেকার" কার্যপ্রবাহের মতো:

১. **মেকার ধাপ**: এজেন্ট প্রাথমিক তথ্য সংগ্রহ করে এবং একটি উত্তর খসড়া তৈরি করে।
২. **চেকার ধাপ**: এজেন্ট অতিরিক্ত অনুসন্ধান করে বিস্তারিত যাচাই বা ফাঁকা পূরণ করে।

নিচে, এজেন্টকে এমন একটি প্রশ্ন করা হয়েছে যা একাধিক গন্তব্যের তুলনা করতে হবে, যার ফলে এজেন্টকে কয়েকবার অনুসন্ধান করতে হচ্ছে।


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## সারাংশ

এই পাঠে আপনি শিখেছেন কীভাবে মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক ব্যবহার করে একটি **এজেন্টিক RAG** সিস্টেম তৈরি করতে হয়:

- **এজেন্টিক RAG** এজেন্টদের স্বতন্ত্রভাবে সিদ্ধান্ত নেওয়ার সুযোগ দেয় কখন তথ্য পুনরুদ্ধার করতে হবে, ফলে পুনরুদ্ধার স্থির না হয়ে গতিশীল হয়।
- **সরঞ্জাম হিসেবে ডাটা উৎস**: বাহ্যিক জ্ঞানভাণ্ডার (যেমন Azure AI Search) সরঞ্জাম হিসাবে আবৃত থাকে যা এজেন্ট ব্যবহার করতে পারে।
- **পর্যায়ক্রমিক পুনরুদ্ধার**: মেকার-চেকার প্যাটার্ন এজেন্টকে একাধিক পুনরুদ্ধার ধাপ সম্পাদনের সুযোগ দেয় — অনুসন্ধান, যাচাইকরণ, এবং পরিমার্জন — চূড়ান্ত উত্তর দেওয়ার আগে।

উৎপাদনে, বড় পরিসরের ভ্রমণ নথি পুনরুদ্ধারের জন্য আপনি ইন-মেমরি `TRAVEL_KNOWLEDGE_BASE` কে প্রকৃত Azure AI Search ইনডেক্স দিয়ে প্রতিস্থাপন করবেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
